# NESG: frozen ESM-2 all-layer extraction
**Reopening:** use Run all to check and resume. Installed packages, completed pooled activations and matching successful pilots are reused. Missing temporary files are restored from private HF; completed activations are not recomputed. Full-map extraction resumes committed batches when its optional cell is present.
It extracts only 2,361 training + 691 validation proteins. Only the development extraction manifest is read; no reserved-test records or phenotype labels are loaded.
Use a GPU runtime (T4 or better), enable notebook access to the Colab Secret **HF_TOKEN_PLM**, and mount the Google Drive containing your existing project folder in the first cell. No ZIP upload is needed. No model fine-tuning, probes or SAE analysis runs here.
Hugging Face is the only persistent activation store. Every completed 64-protein shard is uploaded immediately to your explicitly configured private dataset. Google Drive supplies existing code and input sequences only. They are copied automatically to temporary Colab disk for execution; generated activations and model caches are never written to Drive.

In [1]:
from pathlib import Path
import hashlib, shutil, os, subprocess, sys, json
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/plm-property-interpretability')
source = PROJECT_ROOT / 'nesg_primary/esm_layer_probing'
work = Path('/content/nesg_esm_stage1')  # Temporary execution copy, not Drive.
required = [
    'common.py', 'extract.py', 'hf_storage.py', 'requirements-colab.txt',
    'install_colab_dependencies.py', 'protocol.json',
    'data/extraction_manifest.tsv', 'data/input_integrity.json',
]
missing = [name for name in required if not (source / name).is_file()]
if missing:
    raise FileNotFoundError(f'Project files not found under {source}. Check PROJECT_ROOT and Drive sync. Missing: {missing}')
for name in required:
    src = source / name
    dest = work / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.is_file() or hashlib.sha256(dest.read_bytes()).digest() != hashlib.sha256(src.read_bytes()).digest():
        shutil.copyfile(src, dest)
    assert hashlib.sha256(dest.read_bytes()).digest() == hashlib.sha256(src.read_bytes()).digest()
os.chdir(work)
print('Read existing code and development inputs from:', source)
print('Temporary execution directory:', work)
print('Activation destination: private Hugging Face; no activation files will be saved to Drive.')

Mounted at /content/drive
Read existing code and development inputs from: /content/drive/MyDrive/plm-property-interpretability/nesg_primary/esm_layer_probing
Temporary execution directory: /content/nesg_esm_stage1
Activation destination: private Hugging Face; no activation files will be saved to Drive.


## 1. Check GPU, then prepare a separate extraction environment
The installer first verifies a CUDA-enabled PyTorch build and an actual GPU allocation. A CPU-only build or disagreement between an actual import and its active metadata stops the run before any installation. Inactive duplicate metadata entries are logged and ignored; they do not force a runtime reset. It then creates `/content/nesg_esm_env`, which reads Colab's existing GPU/numerical libraries but keeps the pinned extraction packages separate from the notebook's packages. Only prebuilt wheels are allowed; no NumPy source builds. TensorFlow/Flax are disabled for this torch-only workflow.
On reopening, pinned packages are checked in the dedicated interpreter. Matching installations skip pip; GPU/import checks still run. A deleted runtime needs a fresh environment, but saved HF results remain reusable.

In [2]:
import shlex
from IPython.utils.process import system
os.environ.update(USE_TORCH='1', USE_TF='0', USE_FLAX='0', PYTHONNOUSERSITE='1')
status = system(shlex.join([sys.executable, '-u', 'install_colab_dependencies.py']))
if status != 0:
    raise RuntimeError(f'Dependency setup stopped (exit {status}). See the live output above.')
EXTRACTION_PYTHON = Path('/content/nesg_esm_env/bin/python')
assert EXTRACTION_PYTHON.is_file()
def run_stage1(*arguments):
    status = system(shlex.join([str(EXTRACTION_PYTHON), '-u', *map(str, arguments)]))
    if status != 0:
        raise RuntimeError(f'Stage 1 command stopped (exit {status}). See the live output above.')

[1/3] Checking the existing GPU runtime BEFORE package changes...
GPU preflight passed: {'python': '3.13.16', 'torch': '2.14.1+cu130', 'cuda': '13.0', 'gpu': 'Tesla T4', 'numpy': '2.4.6', 'pandas': '2.2.3', 'psutil': '5.9.5'}
Multiple metadata entries for numpy: ['1.26.4', '2.4.6']; keeping active version 2.4.6 (matches imported version).
[2/3] Installing binary extraction packages into /content/nesg_esm_env
The notebook Python environment is not an installation target; no source builds are allowed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 102.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 27.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.6 MB/s  0:00:00
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.8.0
    Not uninstalling safetensors at /usr/local/lib/python3.13/dist-packages, outside environment /content/nesg_esm_env
    Can't uninstall 'safetensors'. No files were found to u

## 2. Secure authentication and persistent output
The token is retrieved only from Colab Secrets as `HF_TOKEN_PLM`, placed in the same environment variable, and passed directly to authenticated HF calls. It is never displayed or saved. Do not paste credentials into cells. The token needs write access to your configured private dataset in addition to model access. The suggested repository name is not proof that it exists. Leave `CREATE_PRIVATE_REPO=False` to check an existing private dataset; set it to `True` only to explicitly create the configured private dataset if missing. This requires organization creation permission. Public repositories are always refused.

In [3]:
from google.colab import userdata
try:
    secret = userdata.get('HF_TOKEN_PLM')
except Exception:
    raise RuntimeError('Enable notebook access to HF_TOKEN_PLM in Colab Secrets.') from None
if not secret:
    raise RuntimeError('HF_TOKEN_PLM is missing from Colab Secrets.')
os.environ['HF_TOKEN_PLM'] = secret
del secret
DESTINATION_REPO = 'PLM-Interp/nesg-esm2-layerwise'
CREATE_PRIVATE_REPO = False  # Set True only to explicitly create this private dataset if missing.
if not DESTINATION_REPO:
    raise ValueError('Set DESTINATION_REPO to your existing private HF dataset before continuing.')
OUTPUT = Path('/content/nesg_esm_stage1_scratch/activations')
OUTPUT.mkdir(parents=True, exist_ok=True)
CACHE = '/content/nesg_esm_model_cache'  # Outside repository; can redownload after disconnection.
# HF/model libraries run only in the dedicated interpreter, never the notebook kernel.
if CREATE_PRIVATE_REPO:
    run_stage1('hf_storage.py', 'init', '--repo', DESTINATION_REPO, '--create-private')
# Restore also checks authentication/privacy. It skips a verified complete
# local cache and otherwise downloads only missing or damaged files.
run_stage1('hf_storage.py', 'restore', '--repo', DESTINATION_REPO, '--folder', OUTPUT)

completed.json: 8.95kB [00:00, 6.31MB/s]
shard_0000.json: 100%|██████████████████████████| 148/148 [00:00<00:00, 929kB/s]
esm_stage1/shard_0000.npz: 100%|███████████| 10.8M/10.8M [00:01<00:00, 5.99MB/s]
shard_0001.json: 100%|██████████████████████████| 148/148 [00:00<00:00, 904kB/s]
esm_stage1/shard_0001.npz: 100%|███████████| 10.8M/10.8M [00:00<00:00, 13.3MB/s]
shard_0002.json: 100%|██████████████████████████| 148/148 [00:00<00:00, 879kB/s]
esm_stage1/shard_0002.npz: 100%|███████████| 10.8M/10.8M [00:00<00:00, 13.1MB/s]
shard_0003.json: 100%|██████████████████████████| 148/148 [00:00<00:00, 942kB/s]
esm_stage1/shard_0003.npz: 100%|███████████| 10.8M/10.8M [00:00<00:00, 13.3MB/s]
shard_0004.json: 100%|██████████████████████████| 148/148 [00:00<00:00, 881kB/s]
esm_stage1/shard_0004.npz: 100%|███████████| 10.8M/10.8M [00:00<00:00, 13.1MB/s]
shard_0005.json: 100%|██████████████████████████| 148/148 [00:00<00:00, 959kB/s]
esm_stage1/shard_0005.npz: 100%|███████████| 10.8M/10.8M [00:00<00:0

## 3. Real-model correctness pilot
Fixed length quantiles plus all three X-containing proteins are chosen without labels.
Checks cover token identities/counts, special-token exclusion, batching/padding invariance, 33 × 1,280 shapes, finite outputs and mixed precision against FP32 across every layer.
Prespecified mixed tolerances: minimum cosine ≥0.999 and maximum relative L2 error ≤0.05. A failed pilot stops extraction; switch to FP32 rather than relaxing thresholds.
FP32 padding check uses `atol=2e-4, rtol=2e-3`. A matching saved pilot is reused. If all pooled activations are already verified, no extraction pilot or model loading is needed.

In [4]:
PRECISION = 'float16'
BATCH_SIZE = 2
MAX_PADDED_TOKENS = 2048
WORKERS = 0
options = ['--output', str(OUTPUT), '--device', 'cuda', '--precision', PRECISION,
           '--batch-size', str(BATCH_SIZE), '--max-tokens', str(MAX_PADDED_TOKENS),
           '--workers', str(WORKERS), '--cache', CACHE, '--hf-repo', DESTINATION_REPO]
run_stage1('extract.py', 'pilot', *options)
if (OUTPUT / 'pilot.json').is_file():
    display(json.loads((OUTPUT / 'pilot.json').read_text()))
else:
    print('Pooled extraction already complete; pilot rerun skipped.')

SKIP pilot: all 3052 pooled activations already complete and verified; no ESM model loaded.
Pooled extraction already complete; pilot rerun skipped.


## 4. Extract all 33 transformer layers
One forward pass per batch, then float32 residue means; full residue tensors are released immediately. `hidden_states[8]` means output at layer 8, not concatenated layers 1–8. Layer 0 is omitted (optional reference not requested for this run). Layer 33 follows the implementation's final encoder normalization. InterProt uses these same HF indices.
If a session disconnects, rerun setup with the same private repository. Setup restores the uploaded shards into temporary storage and checks their hashes and protein alignment; extraction skips them. Each new shard is uploaded before the next batch of proteins starts. Upload failure stops the run; rerun after correcting connectivity or permissions. Only an unfinished or not-yet-uploaded shard may need recomputation. Reduce batch size on OOM; shard identities stay unchanged. Changing precision or protocol requires a different private repository and pilot; incompatible artifacts are never overwritten.

In [5]:
run_stage1('extract.py', 'extract', *options, '--shard-size', '64')

SKIP extract: all 3052 pooled activations already complete and verified; no ESM model loaded.


In [6]:
import shutil

shutil.copyfile(
    source / "pilot_full_map_precision.py",
    work / "pilot_full_map_precision.py",
)
shutil.copyfile(
    source.parent.parent / "interprot_ref/raw/sae_model.py",
    work / "sae_model.py",
)

run_stage1(
    "pilot_full_map_precision.py",
    "--repo", DESTINATION_REPO,
    "--pooled-dir", OUTPUT,
    "--cache", CACHE,
    "--sae-source", work / "sae_model.py",
)

(…)f7ce698ff048679ee17a1373b817e5ef75e.json: 83.4kB [00:00, 86.0MB/s]
SKIP precision pilot: verified previous PASS for 11 proteins / 9 SAE layers in private HF. No models downloaded or run.


In [7]:
import shutil

shutil.copyfile(
    source / "save_full_maps.py",
    work / "save_full_maps.py",
)

run_stage1(
    "save_full_maps.py",
    "--repo", DESTINATION_REPO,
    "--pooled-dir", OUTPUT,
    "--cache", CACHE,
    "--dtype", "float16",
)

Streaming output truncated to the last 5000 lines.
  ...31c1a1c417d55.safetensors:  53%|███████▍      | 20.0MB / 37.3MB            








Processing Files (0 / 8)      :  36%|█████         | 53.6MB /  148MB, 4.41MB/s  
New Data Upload               :  36%|█████         | 53.6MB /  148MB, 4.41MB/s  

  ...e9c18bc81a7f1.safetensors:  34%|████▋         | 4.74MB / 14.0MB            


  ...1cc61222c80f9.safetensors:  34%|████▋         | 5.74MB / 17.0MB            



  ...683b18b0b9789.safetensors:  35%|████▉         | 6.68MB / 18.8MB            




  ...d82a51f81a4fa.safetensors:  35%|████▉         | 5.09MB / 14.4MB            





  ...c04b4ae8b0f26.safetensors:  34%|████▋         | 7.37MB / 21.8MB            






  ...f67e150093b0f.safetensors:  35%|████▉         | 3.80MB / 10.7MB            







  ...31c1a1c417d55.safetensors:  58%|████████      | 21.5MB / 37.3MB            








Processing Files (0 / 8)      :  40%|█████▋        | 59.9MB /  148MB, 4.87MB/s  
New Data Uploa

## 5. Validate complete output
Expected 3,052 × 33 × 1,280 float32 values (~516 MB total, before metadata).
The final completion manifest contains exact artifact checksums. Experimental labels and Pfam annotations stay in the separate local analysis manifest.

In [8]:
run_stage1('extract.py', 'validate', '--output', OUTPUT)
completed = json.loads((OUTPUT / 'completed.json').read_text())
print('Validated shape:', completed['shape'])
print('Output:', OUTPUT)

Verified finite aligned activations (3052, 33, 1280)
Validated shape: [3052, 33, 1280]
Output: /content/nesg_esm_stage1_scratch/activations


## 6. Verify durable private storage
Extraction uploads every validated shard immediately, then the final completion manifest. This cell rechecks privacy and retrieves the final manifest from the exact uploaded commit to confirm the run reached Hugging Face. Generated activations are not saved to Google Drive.
Uploaded files contain activations and minimal ID/hash/split/extraction metadata, not sequences, experimental labels, Pfam annotations or reserved-test data.

In [9]:
verify_remote = "import json, sys\nfrom pathlib import Path\nfrom common import token\nfrom hf_storage import private_api\nfrom huggingface_hub import hf_hub_download\nrepo, output = sys.argv[1], Path(sys.argv[2])\n_, repository = private_api(repo)\ntry:\n    remote_index = hf_hub_download(repo, 'esm_stage1/completed.json',\n        repo_type='dataset', revision=repository.sha, token=token())\nexcept Exception:\n    raise RuntimeError('Could not verify remote completion; check token/repository/network. Details suppressed.') from None\nassert json.loads(Path(remote_index).read_text()) == json.loads((output/'completed.json').read_text())\nprint('All activations are stored in a verified private HF dataset:', repo)\nprint('Remote commit:', repository.sha)\n"
run_stage1('-c', verify_remote, DESTINATION_REPO, OUTPUT)

All activations are stored in a verified private HF dataset: PLM-Interp/nesg-esm2-layerwise
Remote commit: a5baf8481b03ba82487d156dcca0f13d59ab3e83


## 7. Reuse activations from Hugging Face
The canonical activation copy is the private Hugging Face dataset. To run probes, use `hf_storage.py download --repo PLM-Interp/nesg-esm2-layerwise --folder /tmp/nesg_activations` with `HF_TOKEN_PLM`, then `probe.py --activations /tmp/nesg_activations`. Downloaded arrays are a disposable working cache, not an additional persistent backup.
Existing experimental labels and Pfam annotations remain in the local research repository; they are not newly uploaded. Rerun the analysis notebook after probing. No model download is needed for probing. **Stop after Stage 1.**